In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install -q peft datasets trl transformers accelerate bitsandbytes

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import torch
from datasets import load_dataset

In [ ]:
train_path = '/content/drive/MyDrive/hybrid_ner_extractor/data/processed/train.jsonl'
val_path = '/content/drive/MyDrive/hybrid_ner_extractor/data/processed/val.jsonl'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

In [ ]:
model_name = 'Qwen/Qwen2.5-0.5B'
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
dataset = load_dataset('json', data_files={'train': train_path, 'validation': val_path})

sft_config = SFTConfig(
    output_dir='./llm_finetuned',
    max_length=512,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    max_steps=800,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy='steps',
    save_steps=200,
    eval_strategy='steps',
    eval_steps=50,
    report_to='none',
    dataset_text_field='text'
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation']
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained('/content/drive/MyDrive/hybrid_ner_extractor/models/llm_lora')
tokenizer.save_pretrained('/content/drive/MyDrive/hybrid_ner_extractor/models/llm_lora')